In [17]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import TypedDict, Literal

In [18]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0.5
)

In [19]:
class UserMessage(TypedDict):
    message : str
    sentiment : Literal["positive", "negative"]
    diagnosis : dict
    response : str

In [20]:
class SentimentSchema(BaseModel):
    sentiment : Literal["positive", "negative"]

class DiagonisSentiment(BaseModel):
    problem : Literal["bug", "ui", "network", "support", "other"] = Field(description="user problem in the product")
    tone : Literal["happy", "frustration", "sad"] = Field(description="user tone in the message")
    urgency : Literal["high", "medium", "low"] = Field(description="importance of the issue urgency")

In [21]:
sentiment_model = llm.with_structured_output(SentimentSchema)
diagnosis_model = llm.with_structured_output(DiagonisSentiment)

In [22]:
def sentiment_analysis(state : UserMessage):
    prompt = f""" understand the user message and classify it into neagtive or positive response form the user message {state["message"]}"""
    response = sentiment_model.invoke(prompt).sentiment
    return {
        "sentiment" : response
    }


In [23]:
def diagnosis_analysis(state : UserMessage):
    prompt = f"""understand the user message {state["message"]} and clasify its probelm, tone, urgency """
    response = diagnosis_model.invoke(prompt)
    return {
        "diagnosis" : response.model_dump()
    }

In [24]:
def negative_analysis(state : UserMessage):
    prompt = f""" understand the user message {state["message"]} and its probelm, tone with urgency {state["diagnosis"]} and give a high quality response to the user for their problem"""
    response = llm.invoke(prompt)
    return {
        "response" : response.content
    }

In [25]:
def positive_analysis(state : UserMessage):
    prompt = f""" understand the user message {state["message"]} and give a heart warm happy message \n and also tell them to give rating in on our website"""
    response = llm.invoke(prompt)
    return {
        "response" : response.content
    }

In [26]:
def sentiment_check(state : UserMessage) -> Literal["diagnosis_analysis", "positive_analysis"]:
    sentiment = state["sentiment"]
    if sentiment == "positive":
        return "positive_analysis"
    else:
        return "diagnosis_analysis"


In [27]:
graph = StateGraph(UserMessage)

graph.add_node("sentiment_analysis", sentiment_analysis)
graph.add_node("diagnosis_analysis", diagnosis_analysis)
graph.add_node("negative_analysis", negative_analysis)
graph.add_node("positive_analysis", positive_analysis)

graph.add_edge(START, "sentiment_analysis")
graph.add_conditional_edges("sentiment_analysis", sentiment_check)

graph.add_edge("diagnosis_analysis", "negative_analysis")
graph.add_edge("negative_analysis", END)
graph.add_edge("positive_analysis", END)

workflow = graph.compile()



In [31]:
initial_state = {
    "message": "I have recieved my product its awesome and iam loving it there is a little bit of damage on my product!"
}

final_state = workflow.invoke(initial_state)

print(final_state["sentiment"])

positive
